<a href="https://colab.research.google.com/github/Kaviyarasi-Sasiperumal/AI_-Based_-Document-_Search_-and-_Knowledge-_Retrieval_-with-_Conversational_Interface/blob/main/Milestone_4_Deployment_%26__Final_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf sentence-transformers faiss-cpu transformers gradio torch


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import gradio as gr
import time


In [ ]:
documents = []
metadata = []
embedder = SentenceTransformer("all-MiniLM-L6-v2")
index = None
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)


In [ ]:
def chunk_text(text, chunk_size=250, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])
    return chunks


def load_document(file):
    global documents, metadata, index

    documents = []
    metadata = []

    reader = PdfReader(file.name)

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            chunks = chunk_text(text)
            for chunk in chunks:
                documents.append(chunk)
                metadata.append(f"{file.name} - page {page_num+1}")

    if not documents:
        return "❌ No readable text found in document."


    embeddings = embedder.encode(documents)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return f"✅ Document loaded successfully! Pages indexed: {len(metadata)}"


In [ ]:
!pip install gradio PyPDF2

In [ ]:
import gradio as gr
import PyPDF2
import time

document_text = ""
chunks = []

# -------- CHUNK FUNCTION --------
def create_chunks(text, chunk_size=300):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

# -------- LOAD MULTIPLE DOCUMENTS --------
def load_document(files):
    global document_text, chunks
    document_text = ""
    chunks = []

    if not files:
        return "❌ No documents uploaded", ""

    total_pages = 0

    for file in files:
        text = ""

        if file.name.lower().endswith(".pdf"):
            reader = PyPDF2.PdfReader(file)
            total_pages += len(reader.pages)
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t + "\n"

        elif file.name.lower().endswith(".txt"):
            text += file.read().decode("utf-8")
            total_pages += 1

        document_text += text + "\n"

    chunks = create_chunks(document_text)

    stats = f"""
📊 DOCUMENT STATISTICS

📄 Total Documents : {len(files)}
📄 Total Pages     : {total_pages}
🧩 Total Chunks    : {len(chunks)}
📝 Total Words     : {len(document_text.split())}
🔡 Total Characters: {len(document_text)}
"""
    return "✅ Documents processed successfully", stats


def get_answer_from_document(question):
    lines = [l.strip() for l in document_text.split("\n") if l.strip()]
    q = question.lower().strip()


    field_keys = ["degree", "branch", "specialization", "name", "email", "phone", "qualification"]
    for key in field_keys:
        if key in q:
            for line in lines:
                if line.lower().startswith(key) and ":" in line:
                    return line.split(":", 1)[1].strip()

    if "skill" in q:
        skills = [l.strip("•- ").strip()
                  for l in lines
                  if l.startswith(("•", "-", ""))]
        if skills:
            return ", ".join(skills)


    if q in ["what is ai", "what is the ai", "define ai", "artificial intelligence"]:
        for i, line in enumerate(lines):
            if line.lower().startswith("artificial intelligence"):
                answer = line
                j = i + 1
                while not answer.endswith(".") and j < len(lines):
                    answer += " " + lines[j]
                    j += 1
                return answer

    if "objective" in q:
        for i, line in enumerate(lines):
            if line.lower() == "objective":
                answer = ""
                j = i + 1
                while j < len(lines):
                    answer += lines[j] + " "
                    if lines[j].endswith("."):
                        break
                    j += 1
                return answer.strip()

    return "Sorry, the exact answer was not found in the document."


# -------- CHAT FUNCTION --------
def chat_function(user_input, history):
    start_time = time.perf_counter()
    msg = user_input.lower()

    if msg in ["hi", "hello", "hey"]:
        return "Hello 👋 Upload documents and ask questions."

    if not document_text:
        return "📄 Please upload documents first."


    if "chunk" in msg:
        return f"📦 Total chunks created: {len(chunks)}"

    answer = get_answer_from_document(user_input)
    response_time = round((time.perf_counter() - start_time) * 1000, 2)

    return f"""📄 Answer:
{answer}

⏱ Response Time: {response_time} ms
"""


# -------- UI --------
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 Multi-Document Chatbot")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📂 Upload Multiple Documents")
            file_input = gr.File(label="Upload PDF / TXT", file_count="multiple")

            status = gr.Textbox(label="Status", interactive=False)
            stats_box = gr.Textbox(label="Document Statistics", lines=8, interactive=False)

            file_input.change(
                load_document,
                inputs=file_input,
                outputs=[status, stats_box]
            )

        with gr.Column(scale=3):
            gr.ChatInterface(
                fn=chat_function,
                title="Ask Questions"
            )

demo.launch(share=True)


